# 7b Machine Learning - Part II

In this final part of the exercise, you will attempt to use a trained k-nearest neighbours model to identify the substitution of an unknown benzene derivative. In particular you are provided with two isomers of aminobenzoic acid.
<br>
<br>To begin with this task, you will need to collect some spectra of your own. You may wish to collect data specifically for isomer *x* or *y*, or you may collect data for both.
<br>
<br>Data should be collected using the left-most IR spectrometry in the 1st year (downstairs) lab - all other spectral data were acquired using this instrument.
<br>
<br>For each isomer you investigate, you will need to collect 5 repeated spectra, cleaning the stage throughouly with isopropanol between collections.
<br>
<br>The following data collection parameters should be employed:

| Parameter | Value |
| :-------: | :---: |
| Measurement Mode | % Transmittance |
| Apodization | Happ-Genzel |
| No. of Scans | 10 |
| Resolution/cm-1 | 0.9 |
| Range/cm-1 | 400 - 4000 |

Each spectrum should be exported from the software as a .txt file (*File* --> *Export*). It is recommended that you use the same systematic naming convention employed in the original spectral library.

---

Now that you have your own data to be identified, the entire original spectral library can be used as a training dataset for the k-nearest neighbours algorithm, with your own acquired data constituting the smaller test dataset.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.axes3d as p3
%matplotlib inline
import C317
import os
os.scandir("/content/drive/MyDrive/Raw_IR_Spectra")
import os
os.scandir("/content/drive/MyDrive/my_IR_data")
import sklearn.decomposition

In [9]:
df_train = C317.load_spectra(0)   # n=0 → no PCA
print("Training DataFrame shape:", df_train.shape)
print(df_train.head())

       o-salicylaldehyde  o-1-chloro-2-nitrobenzene  o-salicylic acid  \
630.0           0.000186                   0.000212          0.000170   
631.0           0.000184                   0.000211          0.000166   
632.0           0.000178                   0.000209          0.000168   
633.0           0.000173                   0.000207          0.000164   
634.0           0.000171                   0.000204          0.000162   
...                  ...                        ...               ...   
876.0           0.000218                   0.000257          0.000164   
877.0           0.000202                   0.000257          0.000163   
878.0           0.000179                   0.000258          0.000164   
879.0           0.000156                   0.000257          0.000161   
880.0           0.000139                   0.000256          0.000159   

       o-guaiacol  o-aminobenzylalcohol  m-tolualdehyde  o-bromotoluene  \
630.0    0.000241              0.000148        0

In [13]:
import os
unknown_files = [
    "A_1.txt",
    "A_2.txt",
    "A_3.txt",
    "A_4.txt",
    "A_5.txt",
]

unknown_path = "/content/drive/MyDrive/my_IR_data"

unknown_dfs = []
for fname in unknown_files:
    fpath = os.path.join(unknown_path, fname)
    df = pd.read_csv(
        fpath,
        skiprows=4,
        sep=r"\s+",
        names=[fname],
        index_col=0
    )
    df = C317.normalisation(df)
    df = C317.interpolation(df)
    df = C317.narrow_range(df)
    # Strip the numeric suffix and .txt so column names match the convention
    base_name = re.sub(r'_\d+$', '', fname.removesuffix('.txt'))
    df.columns = [base_name]
    unknown_dfs.append(df)

df_test = pd.concat(unknown_dfs, axis=1)
print("Test DataFrame shape:", df_test.shape)
print(df_test.head())

Test DataFrame shape: (251, 5)
              A         A         A         A         A
630.0  0.000304  0.000295  0.000290  0.000340  0.000283
631.0  0.000315  0.000290  0.000301  0.000342  0.000269
632.0  0.000308  0.000297  0.000293  0.000329  0.000269
633.0  0.000295  0.000309  0.000294  0.000326  0.000269
634.0  0.000305  0.000302  0.000286  0.000330  0.000265


In [14]:
labels = [col[0] for col in df_train.columns]
print("Labels:", labels)

Labels: ['o', 'o', 'o', 'o', 'o', 'm', 'o', 'm', 'p', 'm', 'o', 'm', 'o', 'o', 'o', 'p', 'o', 'o', 'o', 'm', 'm', 'p', 'm', 'p', 'o', 'm', 'm', 'm', 'o', 'o', 'p', 'o', 'o', 'o', 'o', 'p', 'p', 'm', 'o', 'p', 'm', 'm', 'p', 'm', 'm', 'm', 'o', 'o', 'm', 'p', 'o', 'm', 'm', 'm', 'o', 'm', 'o', 'm', 'm', 'p', 'o', 'o', 'm', 'p', 'o', 'p', 'm', 'p', 'o', 'm', 'p', 'o', 'm', 'm', 'm', 'o', 'm', 'o', 'm', 'm', 'o', 'o', 'p', 'p', 'p', 'p', 'o', 'p', 'o', 'm', 'o', 'o', 'm', 'o', 'm', 'p', 'm', 'm', 'm', 'o', 'p', 'm', 'm', 'o', 'm', 'm', 'm', 'm', 'o', 'p', 'p', 'o', 'p', 'o', 'o', 'o', 'm', 'm', 'o', 'o', 'm', 'm', 'o', 'm', 'p', 'm', 'm', 'm', 'm', 'o', 'p', 'p', 'm', 'o', 'o', 'o', 'm', 'm', 'o', 'm', 'o', 'p', 'm', 'o', 'p', 'p', 'p', 'p', 'o', 'm', 'o', 'o', 'p', 'p', 'm', 'm', 'p', 'o', 'o', 'o', 'o', 'm', 'm', 'm', 'm', 'm', 'p', 'o', 'p', 'o', 'm', 'o', 'p', 'p', 'm', 'm', 'o', 'm', 'p', 'm', 'o', 'p', 'p', 'm', 'o', 'm', 'o', 'o', 'm', 'm', 'o', 'o', 'm', 'p', 'o', 'p', 'm', 'm', '

In [15]:
X_train = df_train.T.values
y_train = labels

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("k-NN model trained.")

k-NN model trained.


In [16]:
X_test = df_test.T.values

predictions = knn.predict(X_test)

for fname, pred in zip(unknown_files, predictions):
    print(f"{fname}  →  predicted class: {pred}-aminobenzoic acid")

A_1.txt  →  predicted class: o-aminobenzoic acid
A_2.txt  →  predicted class: p-aminobenzoic acid
A_3.txt  →  predicted class: o-aminobenzoic acid
A_4.txt  →  predicted class: p-aminobenzoic acid
A_5.txt  →  predicted class: o-aminobenzoic acid


---